# Machining Sounds

## Part B: Sound

*A FAB26 workshop lesson, companion to* Machining Dynamics: Jupyter Notebook Edition *(DOI: forthcoming)*

**Authors:** Tony L. Schmitz and Michael F. Gomez

> *"The profound study of nature is the most fertile source of mathematical discoveries."*
> Joseph Fourier, *The Analytical Theory of Heat* (1822)


Part A ended with a debt. The microphone records only pressure against time, yet the tap test of Sect. A.7 and the three views of Sect. A.8 displayed spectra, frequency axes produced by a function we used without opening. Part B pays that debt. Starting from what a microphone actually measures, we follow the chain that turns a recording into a spectrum: the recording becomes a list of numbers, a periodic signal becomes a sum of sinusoids, and the discrete Fourier transform connects the two. By the end, the `compute_spectrum` and `compute_spectrogram` functions that produced the figures of Part A are short, inspectable computations rather than trusted tools.

The purposes of Part B are to:

- Describe sound as a traveling pressure wave and identify what a microphone records.
- Explain sampling, the sample rate, the Nyquist frequency, and aliasing.
- Develop the Fourier series and the idea that a periodic signal is a recipe of harmonics.
- Define the discrete Fourier transform, its fast algorithm, and its frequency resolution.
- Build the spectrogram from short-time spectra and examine its time-frequency tradeoff.
- Apply the complete toolkit to a recording of your own, choosing the sampling and analysis parameters.
- Connect a periodic force to a harmonic sound, the observation that carries us into Part C.

Each concept is demonstrated interactively: run the cells, listen, drag a slider, and run again. Part A is assumed; nothing else is.


### Nomenclature

The following symbols are used consistently throughout the lesson:

| Symbol | Meaning | Units |
|--------|---------|-------|
| $t$ | time | s |
| $f$ | frequency | Hz |
| $c$ | speed of sound in air | m/s |
| $f_s$ | sample rate | Hz |
| $f_N$ | Nyquist frequency, $f_s / 2$ | Hz |
| $N$ | number of samples in a record | dimensionless |
| $T$ | record length, $N / f_s$ | s |
| $\Delta f$ | frequency resolution, $1 / T$ | Hz |
| $f_0$ | fundamental frequency of a periodic signal | Hz |
| $k$ | harmonic index | dimensionless |
| $A_k$, $\phi_k$ | amplitude and phase of the $k$-th harmonic | signal units, rad |
| $x_n$ | the $n$-th sample of a recording | signal units |
| $X_m$ | the $m$-th coefficient of the DFT | signal units |

In the code cells, $f_s$ appears as the constant `AUDIO_RATE`, matching Part A; other symbols keep their names (`f0`, `N`, `df`).


## Setup: Install and Import Libraries

Run the install cell once. If this is the first install on your machine, restart the kernel afterward (Kernel menu, Restart), then continue from the import cell.


In [ ]:
%pip install -q numpy matplotlib plotly


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from IPython.display import Audio, display

AUDIO_RATE = 48000  # Hz, sample rate for all synthesized sound

print('Libraries loaded.')


The utility functions used throughout the lesson are defined in the collapsed cell below and documented in the Appendix. They are the same functions used in Part A.


In [ ]:
# --- Utility functions used throughout the lesson (documented in the Appendix) ---
def fade(x, ms=15, fs=AUDIO_RATE):
    """Apply a short raised-cosine fade-in and fade-out so audio clips start
    and stop without clicks."""
    n = int(ms * 1e-3 * fs)
    ramp = 0.5 * (1 - np.cos(np.pi * np.arange(n) / n))
    y = x.copy()
    y[:n] *= ramp
    y[-n:] *= ramp[::-1]
    return y

def compute_spectrum(x, fs, fmax=4000):
    """Amplitude spectrum of signal x on a linear scale, normalized to a peak
    value of one. Returns the frequency vector (Hz, limited to fmax) and the
    normalized spectrum."""
    x = np.asarray(x, dtype=float)
    w = np.hanning(len(x))
    X = np.fft.rfft((x - x.mean()) * w)
    f = np.fft.rfftfreq(len(x), 1 / fs)
    amp = 2 * np.abs(X) / np.sum(w)
    amp = amp / amp.max()
    sel = f <= fmax
    return f[sel], amp[sel]

def compute_spectrogram(x, fs, nfft=2048, hop=512, fmax=4000):
    """Short-time amplitude spectrogram on a linear scale, normalized to a
    peak value of one. Returns (t_frames, f, S) where S has one column per
    time frame and one row per frequency."""
    w = np.hanning(nfft)
    n_frames = 1 + max(0, (len(x) - nfft) // hop)
    f = np.fft.rfftfreq(nfft, 1 / fs)
    sel = f <= fmax
    S = np.empty((int(sel.sum()), n_frames))
    for j in range(n_frames):
        S[:, j] = np.abs(np.fft.rfft(x[j * hop:j * hop + nfft] * w))[sel]
    S /= S.max()
    t_frames = (np.arange(n_frames) * hop + nfft / 2) / fs
    return t_frames, f[sel], S

def decimate_for_plot(t, x, max_points=None):
    """Pass plotting data through unchanged by default (max_points=None).
    To thin a long record for display, set max_points to an integer: each
    output bin then keeps its local minimum and maximum, preserving the
    visual envelope without aliasing. Computations always use the
    full-rate data."""
    n = len(x)
    if max_points is None or n <= max_points:
        return np.asarray(t), np.asarray(x)
    bins = max_points // 2
    edge = (n // bins) * bins
    xb = np.asarray(x)[:edge].reshape(bins, -1)
    tb = np.asarray(t)[:edge].reshape(bins, -1)
    tt = np.repeat(tb.mean(axis=1), 2)
    xx = np.empty(2 * bins)
    xx[0::2] = xb.min(axis=1)
    xx[1::2] = xb.max(axis=1)
    return tt, xx


---

## B.1 Sound as a Pressure Wave

A vibrating surface pushes on the air ahead of it as it moves forward and leaves a slight rarefaction as it moves back. Each layer of air nudges the next, and the disturbance travels outward at the speed of sound, about 343 m/s in room-temperature air. As Sect. A.1 recounted, Newton computed this speed from the air's pressure and density in 1687, and Laplace corrected the missing thermal effect in 1816. The traveling disturbance is a *pressure wave*: at a fixed point in the room, the air pressure rises and falls slightly about its resting value as the wave passes.

A microphone measures exactly that. Its diaphragm rides the pressure fluctuation, and the recording is a single number, pressure, reported over and over as time advances. Two properties of the wave map onto what we hear. The rate of the fluctuation sets the *pitch*: human hearing spans roughly 20 Hz to 20 kHz, and faster fluctuation is heard as higher pitch. The size of the fluctuation sets the *loudness*.

Look closely at what the recording contains, though: pressure against time, and nothing else. No frequency axis, no peaks, no spectrum. Every frequency-domain figure in Part A was computed from a record of exactly this kind, and the rest of Part B follows that computation from one end to the other. The first step is unavoidable: a computer cannot hold a continuous wave, so the recording must first become a list of numbers.

▸ **Key terms: pressure wave, speed of sound, pitch, loudness**


---

## B.2 Sampling

A phone converts the continuous pressure signal into numbers by measuring it at regular intervals: $f_s$ times per second, the *sample rate*. Each measurement is a *sample*, and the recording becomes the list $x_0, x_1, x_2, \ldots$, with the true wave discarded between samples. The measurement is honest as long as the wave does not wiggle appreciably between samples, and the boundary is quantified by the sampling theorem of Nyquist and Shannon [1, 2]: a sample rate of $f_s$ faithfully represents frequencies up to the *Nyquist frequency*

$$f_N = \frac{f_s}{2} \tag{B.1}$$

and no higher. Phones sample sound at $f_s = 48{,}000$ per second, so $f_N = 24$ kHz sits comfortably above the limit of human hearing, which is why a phone recording can capture any audible sound. The explorer of Fig. B.1 samples a 440 Hz tone at an adjustable rate. The dashed curve is the lowest-frequency wave consistent with the samples, which is the wave any playback or analysis will report. At rates above twice the tone's frequency it overlays the tone itself, and the samples are a faithful record; drag the slider below 880 per second and the two part company.


In [ ]:
# --- Fig. B.1: a tone sampled at an adjustable rate ---
f_tone = 440.0
t_fine = np.arange(int(0.01 * AUDIO_RATE)) / AUDIO_RATE            # 10 ms at 48 kHz
FS_STEPS = [500, 600, 700, 800, 1000, 1500, 3000, 8000]
START_FS = 6   # slider starts at 3,000 per second, above the Nyquist limit

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_fine * 1e3, y=np.sin(2 * np.pi * f_tone * t_fine),
                         mode='lines', line=dict(color='#1f77b4', width=1),
                         name='pressure wave, 440 Hz'))
for i, fs_i in enumerate(FS_STEPS):
    t_s = np.arange(int(0.01 * fs_i) + 1) / fs_i
    d = f_tone - fs_i * round(f_tone / fs_i)     # signed frequency implied by the samples
    vis = (i == START_FS)
    fig.add_trace(go.Scatter(x=t_s * 1e3, y=np.sin(2 * np.pi * f_tone * t_s),
                             mode='markers', marker=dict(color='#d62728', size=8),
                             name='samples', visible=vis))
    fig.add_trace(go.Scatter(x=t_fine * 1e3, y=np.sin(2 * np.pi * d * t_fine),
                             mode='lines', line=dict(color='#ff7f0e', dash='dash'),
                             name=f'lowest-frequency wave through the samples, {abs(d):.0f} Hz',
                             visible=vis))

steps = []
for i, fs_i in enumerate(FS_STEPS):
    vis = [True] + [False] * (2 * len(FS_STEPS))
    vis[1 + 2 * i:3 + 2 * i] = [True, True]
    steps.append(dict(method='update', label=f'{fs_i:,}', args=[{'visible': vis}]))

fig.update_layout(
    sliders=[dict(active=START_FS, currentvalue=dict(prefix='samples per second = '),
                  steps=steps)],
    title='Fig. B.1 — A 440 Hz tone sampled at an adjustable rate',
    template='simple_white', height=430,
    xaxis_title='t (ms)', yaxis_title='pressure (normalized)',
    legend=dict(x=0.98, y=0.98, xanchor='right'),
    margin=dict(l=60, r=20, t=60, b=90))
fig.show()


Below the Nyquist limit the samples still exist, but they no longer pin down the wave that produced them: at 600 samples per second, the explorer shows the 440 Hz tone producing exactly the samples of a 160 Hz wave. This failure is *aliasing*, and the recording that results reports the slower, aliased signal in place of the true one. Fig. B.2 fixes one case for listening: a 3,200 Hz tone sampled at 4,000 per second produces exactly the samples of an 800 Hz tone.


In [ ]:
# --- Fig. B.2: aliasing of a tone above the Nyquist frequency ---
f_true = 3200.0
fs_slow = 4000
t_fine = np.arange(int(0.005 * AUDIO_RATE)) / AUDIO_RATE           # 5 ms
t_samp = np.arange(int(0.005 * fs_slow)) / fs_slow
f_alias = abs(f_true - fs_slow * round(f_true / fs_slow))          # 800 Hz

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_fine * 1e3, y=np.sin(2 * np.pi * f_true * t_fine),
                         mode='lines', line=dict(color='#1f77b4', width=1),
                         name=f'true tone, {f_true:.0f} Hz'))
fig.add_trace(go.Scatter(x=t_fine * 1e3, y=-np.sin(2 * np.pi * f_alias * t_fine),
                         mode='lines', line=dict(color='#ff7f0e', dash='dash'),
                         name=f'aliased signal, {f_alias:.0f} Hz'))
fig.add_trace(go.Scatter(x=t_samp * 1e3, y=np.sin(2 * np.pi * f_true * t_samp),
                         mode='markers', marker=dict(color='#d62728', size=8),
                         name='samples at 4,000 per second'))
fig.update_layout(title='Fig. B.2 — Aliasing of a 3,200 Hz tone sampled at 4,000 per second',
                  template='simple_white', height=380,
                  xaxis_title='t (ms)', yaxis_title='pressure (normalized)',
                  legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
                  margin=dict(l=60, r=20, t=60, b=50))
fig.show()


Both curves pass through every sample, and the sampled record contains no information to choose between them. Aliasing is audible as well as visible. The first clip below is the true 3,200 Hz tone. The second is what a system sampling at 4,000 per second retains of it, played back crudely: the dominant tone is the aliased signal at 800 Hz.


In [ ]:
# --- Hearing aliasing ---
dur = 1.2
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
true_tone = 0.5 * np.sin(2 * np.pi * 3200 * t)

fs_slow = 4000
t_slow = np.arange(int(dur * fs_slow)) / fs_slow
sampled = 0.5 * np.sin(2 * np.pi * 3200 * t_slow)
crude = np.repeat(sampled, AUDIO_RATE // fs_slow)   # each sample held until the next

print('The true tone, 3,200 Hz:')
display(Audio(fade(true_tone), rate=AUDIO_RATE))
print('The aliased recording (dominant aliased tone at 800 Hz, plus playback artifacts):')
display(Audio(fade(crude), rate=AUDIO_RATE))


<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
Audio equipment settles this problem by choice of sample rate. Compact discs use 44,100 samples per second and phones use 48,000, both selected so the Nyquist frequency clears the roughly 20 kHz limit of human hearing, and recording hardware filters out content above $f_N$ before sampling so nothing aliases into the record. Although Eq. B.1 requires only that $f_s$ exceed twice the highest frequency of interest, measurement practice is more generous: for vibration work we typically sample at roughly ten times that frequency, so the recorded waveform is smooth on screen as well as recoverable in principle. For machine listening the margins are generous either way: tool natural frequencies, spindle rotation rates, and tooth passing rhythms rarely exceed a few kHz, and a phone samples at ten times that comfortably. Every recording in this lesson series uses $f_s = 48{,}000$.
</div>

▸ **Key terms: sample, sample rate, Nyquist frequency, aliasing**


---

## B.3 The Fourier Series

The recording is now a list of numbers, and the question from Sect. B.1 stands: where does frequency come from? The opening is a simple observation about the sounds that have pitch. A sustained tone from a string, a pipe, or a humming machine produces a pressure signal that repeats: some pattern, of whatever shape, recurring every $T_0$ seconds. The repetition rate $f_0 = 1/T_0$ is the *fundamental frequency*, and it is what the ear reports as the pitch.

Fourier's assertion of 1822, with the precise conditions later supplied by Dirichlet, is that any such *periodic signal* can be written as a sum of sinusoids at integer multiples of the fundamental [3]:

$$x(t) = A_0 + \sum_{k=1}^{\infty} A_k \sin\!\left(2\pi k f_0 t + \phi_k\right) \tag{B.2}$$

The component at $k f_0$ is the $k$-th *harmonic*, and the list of amplitudes $A_1, A_2, A_3, \ldots$ is the recipe language we used informally in Part A: which harmonics are present, and how strongly. Equation B.2 is a synthesis statement. It says a periodic signal can be assembled from its recipe, and the assembly can be watched and heard one term at a time. The explorer below builds a sawtooth-like wave whose recipe is $A_k = 1/k$; drag the slider to add harmonics.


In [ ]:
# --- Fig. B.3: a periodic signal assembled from its harmonics ---
f0 = 250.0
K_STEPS = [1, 2, 3, 4, 5, 8, 12, 20, 40]
START_K = 2   # slider starts at K = 3

t_two = np.arange(int(2 / f0 * AUDIO_RATE)) / AUDIO_RATE           # two periods

fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45],
                    subplot_titles=('Waveform (two periods)', 'Recipe used so far'))
for i, K in enumerate(K_STEPS):
    xK = np.zeros_like(t_two)
    for k in range(1, K + 1):
        xK += (1 / k) * np.sin(2 * np.pi * k * f0 * t_two)
    vis = (i == START_K)
    fig.add_trace(go.Scatter(x=t_two * 1e3, y=xK, mode='lines',
                             line=dict(color='#1f77b4'), showlegend=False,
                             visible=vis), row=1, col=1)
    fig.add_trace(go.Bar(x=[k * f0 for k in range(1, K + 1)],
                         y=[1 / k for k in range(1, K + 1)],
                         width=30, marker_color='#1f77b4', showlegend=False,
                         visible=vis), row=1, col=2)

steps = []
for i, K in enumerate(K_STEPS):
    vis = [False] * (2 * len(K_STEPS))
    vis[2 * i:2 * i + 2] = [True, True]
    steps.append(dict(method='update', label=str(K), args=[{'visible': vis}]))

fig.update_layout(
    sliders=[dict(active=START_K, currentvalue=dict(prefix='harmonics K = '), steps=steps)],
    title='Fig. B.3 — A periodic signal assembled from its harmonics',
    template='simple_white', height=420,
    margin=dict(l=60, r=20, t=70, b=90))
fig.update_xaxes(title_text='t (ms)', row=1, col=1)
fig.update_xaxes(title_text='frequency (Hz)', range=[0, 41 * f0], row=1, col=2)
fig.update_yaxes(title_text='x', row=1, col=1)
fig.update_yaxes(title_text='amplitude A_k', range=[0, 1.05], row=1, col=2)
fig.show()


The convergence is audible. Each clip below is the same partial sum at a fixed number of harmonics: one harmonic is a pure tone, a few harmonics add buzz, and by forty the timbre has stopped changing to most ears, since the remaining harmonics are weak and high. The pitch is identical in every clip, because the fundamental never moves.


In [ ]:
# --- Hearing the partial sums converge ---
dur = 1.2
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
for K in [1, 3, 10, 40]:
    xK = np.zeros_like(t)
    for k in range(1, K + 1):
        xK += (1 / k) * np.sin(2 * np.pi * k * 250 * t)
    xK *= 0.5 / np.max(np.abs(xK))
    print(f'K = {K} harmonic{"s" if K > 1 else ""}:')
    display(Audio(fade(xK), rate=AUDIO_RATE))


### Timbre

Equation B.2 also explains why instruments sound different on the same note. Ohm proposed in 1843 that the ear decomposes a sound into its harmonics, and Helmholtz established the case in 1863: the pitch is set by the fundamental, and the *timbre*, the recognizable character of the sound, is set by the recipe of harmonic amplitudes [4]. Two signals with the same $f_0$ and different recipes have the same pitch and different voices. Fig. B.4 compares two: the $1/k$ recipe from Fig. B.3 with every harmonic present, and a recipe with the same amplitudes on the odd harmonics only.


In [ ]:
# --- Fig. B.4: two recipes at the same pitch ---
f0 = 250.0
dur = 1.2
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
t_two = np.arange(int(2 / f0 * AUDIO_RATE)) / AUDIO_RATE

recipes = [('every harmonic, 1/k', range(1, 9), '#1f77b4', 'solid'),
           ('odd harmonics, 1/k', range(1, 9, 2), '#ff7f0e', 'dash')]

fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45],
                    subplot_titles=('Waveform (two periods)', 'Spectrum'))
sounds = {}
for name, ks, color, dash in recipes:
    x_sig = np.zeros_like(t)
    x_two = np.zeros_like(t_two)
    for k in ks:
        x_sig += (1 / k) * np.sin(2 * np.pi * k * f0 * t)
        x_two += (1 / k) * np.sin(2 * np.pi * k * f0 * t_two)
    sounds[name] = 0.5 * x_sig / np.max(np.abs(x_sig))
    fig.add_trace(go.Scatter(x=t_two * 1e3, y=x_two, mode='lines',
                             line=dict(color=color, dash=dash), name=name),
                  row=1, col=1)
    f_sp, a_sp = compute_spectrum(x_sig, AUDIO_RATE, fmax=2500)
    fig.add_trace(go.Scatter(x=f_sp, y=a_sp, mode='lines',
                             line=dict(color=color, dash=dash), showlegend=False),
                  row=1, col=2)
fig.update_layout(title='Fig. B.4 — Two recipes at the same pitch',
                  template='simple_white', height=400,
                  legend=dict(x=0.44, y=0.98, xanchor='right'),
                  margin=dict(l=60, r=20, t=70, b=50))
fig.update_xaxes(title_text='t (ms)', row=1, col=1)
fig.update_xaxes(title_text='frequency (Hz)', row=1, col=2)
fig.update_yaxes(title_text='x', row=1, col=1)
fig.update_yaxes(title_text='amplitude (normalized)', range=[0, 1.05], row=1, col=2)
fig.show()

for name in sounds:
    print(name + ':')
    display(Audio(fade(sounds[name]), rate=AUDIO_RATE))


Both clips sit at 250 Hz, and they are easy to tell apart. The odd-harmonic recipe has the hollow character of a clarinet-like tone, and the full recipe is the brighter buzz. This observation matters for machine listening: a milling machine has a voice for the same reason an instrument does, a repeating process with a particular recipe of harmonics, and changes in the recipe are audible even when the pitch is unchanged.

### Drawing with Harmonics

Equation B.2 applies to anything periodic, and a closed curve traced in the plane is periodic in its own way: travel around the outline and the path repeats. Writing the path as a complex number $z(u) = x(u) + i\,y(u)$ over one lap, the same synthesis sum reconstructs the outline from a recipe of rotating components, and truncating the recipe at $K$ harmonics gives a smoothed approximation that sharpens as $K$ grows. The explorer below applies this to the outlines of the letters FAB26. Watch where the reconstruction struggles longest: the sharp corners, which demand the high harmonics, for the same reason abrupt sounds carry strong high-frequency content.


In [ ]:
# --- Fig. B.5: the letters FAB26 traced by partial Fourier sums ---
tp = TextPath((0, 0), 'FAB26', size=100,
              prop=FontProperties(family='DejaVu Sans', weight='bold'))
contours = [p for p in tp.to_polygons() if len(p) >= 4]

def resample_contour(poly, M=400):
    d = np.linalg.norm(np.diff(poly, axis=0), axis=1)
    s = np.concatenate([[0], np.cumsum(d)])
    u = np.linspace(0, s[-1], M, endpoint=False)
    return np.interp(u, s, poly[:, 0]) + 1j * np.interp(u, s, poly[:, 1])

def partial_sum(z, K):
    M = len(z)
    C = np.fft.fft(z) / M
    ks = np.fft.fftfreq(M, 1 / M).astype(int)
    n = np.arange(M)
    zr = np.zeros(M, dtype=complex)
    for i in np.where(np.abs(ks) <= K)[0]:
        zr += C[i] * np.exp(2j * np.pi * ks[i] * n / M)
    return zr

Z = [resample_contour(p) for p in contours]
K_IMG = [1, 2, 3, 5, 8, 12, 20, 40, 80]
START_IMG = 2   # slider starts at K = 3

fig = go.Figure()
# the target outline, always visible
gx, gy = [], []
for z in Z:
    gx += list(np.append(z.real, z.real[0])) + [None]
    gy += list(np.append(z.imag, z.imag[0])) + [None]
fig.add_trace(go.Scatter(x=gx, y=gy, mode='lines',
                         line=dict(color='#7f7f7f', width=1), opacity=0.35,
                         hoverinfo='skip', showlegend=False))
for i, K in enumerate(K_IMG):
    rx, ry = [], []
    for z in Z:
        zr = partial_sum(z, K)
        rx += list(np.append(zr.real, zr.real[0])) + [None]
        ry += list(np.append(zr.imag, zr.imag[0])) + [None]
    fig.add_trace(go.Scatter(x=rx, y=ry, mode='lines',
                             line=dict(color='#1f77b4', width=2),
                             visible=(i == START_IMG), showlegend=False))

steps = []
for i, K in enumerate(K_IMG):
    vis = [True] + [False] * len(K_IMG)
    vis[1 + i] = True
    steps.append(dict(method='update', label=str(K), args=[{'visible': vis}]))

fig.update_layout(
    sliders=[dict(active=START_IMG, currentvalue=dict(prefix='harmonics per contour K = '),
                  steps=steps)],
    title='Fig. B.5 — The letters FAB26 traced by partial Fourier sums',
    template='simple_white', height=420,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor='x', scaleratio=1),
    margin=dict(l=20, r=20, t=60, b=90))
fig.show()


<div style="background-color: #e8f4fd; border-left: 5px solid #2196F3; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>Let's Talk About: Corners and Clicks</strong><br>
At a few harmonics per contour, every letter is a rounded blob; the straight strokes arrive early, and the corners arrive last. The audio version of a corner is a click or an abrupt attack: any feature that changes quickly, in space or in time, requires high harmonics to represent. This is one reason the hard hammer tip of Sect. A.7 excited such a wide frequency range: its short, abrupt pulse is the temporal cousin of a sharp corner.
</div>

▸ **Key terms: periodic signal, fundamental frequency, harmonic, Fourier series, timbre**


---

## B.4 The DFT and the FFT

Equation B.2 synthesizes a signal from its recipe. Measurement runs the other direction: given a recorded signal, find the recipe. For a sampled record of $N$ numbers, that inverse operation is the *discrete Fourier transform* (DFT):

$$X_m = \sum_{n=0}^{N-1} x_n \, e^{-i 2\pi m n / N} \tag{B.3}$$

The DFT takes the $N$ samples $x_n$ and returns one coefficient per frequency bin. Bin $m$ corresponds to the frequency

$$f_m = \frac{m f_s}{N} \tag{B.4}$$

and $|X_m|$ reports how strongly the record contains that frequency. The transform loses nothing, because the samples are recovered exactly by the inverse DFT:

$$x_n = \frac{1}{N} \sum_{m=0}^{N-1} X_m \, e^{+i 2\pi m n / N} \tag{B.5}$$

which is the discrete counterpart of the synthesis sum of Eq. B.2; analysis and synthesis are the two directions of one relationship. From Eq. B.4, adjacent bins are separated by the *frequency resolution*, set by the record length $T = N / f_s$:

$$\Delta f = \frac{f_s}{N} = \frac{1}{T} \tag{B.6}$$

A one-second record resolves frequencies 1 Hz apart; a tenth of a second resolves 10 Hz. Longer records buy finer frequency detail, a tradeoff we will meet again in Sect. B.5. Finally, for a sinusoidal component landing in bin $m$, the amplitude follows from the coefficient as

$$A_m = \frac{2\,|X_m|}{\sum_n w_n} \tag{B.7}$$

where $w_n$ is the window discussed below; without a window ($w_n = 1$) the denominator is simply $N$. Computed directly, Eq. B.3 costs about $N^2$ multiplications, which for a one-second phone recording is over two billion. The *fast Fourier transform* (FFT) of Cooley and Tukey computes the identical result in about $N \log_2 N$ operations [5], roughly a thousandfold saving at that size, and it is the reason the spectrum is a routine, real-time computation. In the code, `np.fft.rfft` is the FFT, specialized to real-valued inputs.

### The compute_spectrum Function

The spectra of Part A, the middle column of Fig. A.5 and the force spectra of Fig. A.2 among them, were produced by `compute_spectrum`, and it can now be read line by line:

```python
def compute_spectrum(x, fs, fmax=4000):
    x = np.asarray(x, dtype=float)
    w = np.hanning(len(x))                    # window: taper the record ends
    X = np.fft.rfft((x - x.mean()) * w)       # Eq. B.3, by the FFT
    f = np.fft.rfftfreq(len(x), 1 / fs)       # the bin frequencies of Eq. B.4
    amp = 2 * np.abs(X) / np.sum(w)           # amplitude by Eq. B.7
    amp = amp / amp.max()                     # normalize the peak to one
    sel = f <= fmax
    return f[sel], amp[sel]
```

One line deserves a comment. The *window* `w` tapers the record smoothly to zero at both ends before the transform. The DFT treats the record as one period of a repeating signal, and a record whose ends do not match creates an artificial jump at the seam; the taper removes the jump, at the cost of slightly widening each spectral peak. Every spectrum in this lesson series uses this window.

With the machinery open, we can test it against a known answer. The recipe of Fig. B.3 was $A_k = 1/k$; the cell below synthesizes that signal, computes its spectrum, and reads the amplitudes back at each harmonic.


In [ ]:
# --- Reading a known recipe back from its spectrum ---
f0 = 250.0
dur = 1.0
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
x_sig = np.zeros_like(t)
for k in range(1, 9):
    x_sig += (1 / k) * np.sin(2 * np.pi * k * f0 * t)

f_sp, a_sp = compute_spectrum(x_sig, AUDIO_RATE, fmax=2500)
df = AUDIO_RATE / len(t)
print(f'Record length T = {dur:.1f} s, so the resolution of Eq. B.6 is {df:.1f} Hz per bin.')
print()
print('harmonic   frequency   recipe A_k   measured amplitude')
for k in range(1, 9):
    idx = np.argmin(np.abs(f_sp - k * f0))
    print(f'   {k}        {k * f0:6.0f} Hz     {1 / k:.3f}        {a_sp[idx]:.3f}')


The measured amplitudes reproduce the recipe, normalized so the fundamental reads one. The spectrum is analysis in exactly the sense that Eq. B.2 is synthesis, and the frequency axes of Part A are the output of this computation and nothing else.

The speed of the FFT is worth seeing once. The cell below transforms a quarter-million-sample record and reports the time.


In [ ]:
# --- The cost of a spectrum ---
import time
N = 2**18                     # 262,144 samples, about 5.5 s of audio
x_big = np.sin(2 * np.pi * 440 * np.arange(N) / AUDIO_RATE)
t0 = time.perf_counter()
np.fft.rfft(x_big)
t1 = time.perf_counter()
print(f'FFT of N = {N:,} samples: {(t1 - t0) * 1e3:.1f} ms')
print(f'The direct sum of Eq. B.3 needs about N^2 = {N**2:.1e} multiplications;')
print(f'the FFT needs about N log2 N = {N * np.log2(N):.1e}, a factor of '
      f'{N / np.log2(N):,.0f} fewer.')


▸ **Key terms: discrete Fourier transform (DFT), fast Fourier transform (FFT), frequency resolution, window**


---

## B.5 The Spectrogram

The DFT of a whole record answers one question well, which frequencies the record contains overall, and Sect. A.8 showed its blind spot: the spectrum of the musical scale reported every note and nothing about their order. The spectrogram recovers time by dividing the record into short *frames*, computing a spectrum for each, and stacking the results side by side, each frame advanced from the last by a fixed *hop*. The `compute_spectrogram` utility does precisely this: a loop over frames, one windowed FFT per frame, and the columns assembled into the heatmap image.

Equation B.6 sets the price. Each frame is short, so each frame's resolution is coarse: a 2,048-sample frame at 48,000 samples per second spans 43 ms and resolves about 23 Hz. Shrinking the frame sharpens the timing and blurs the frequencies; growing the frame does the reverse. This *time-frequency tradeoff* is a property of the analysis itself, and the frame length selects a position on it. The explorer below shows the same musical scale from Sect. A.8 analyzed at three frame lengths; drag the slider between them.


In [ ]:
# --- Fig. B.6: one scale, three frame lengths ---
up = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
notes = up + up[-2::-1]
dur_scale = 3.0
note_len = int(dur_scale / len(notes) * AUDIO_RATE)
scale_sig = np.zeros(len(notes) * note_len)
for j, f_note in enumerate(notes):
    seg = np.arange(note_len) / AUDIO_RATE
    scale_sig[j * note_len:(j + 1) * note_len] = fade(
        0.6 * np.sin(2 * np.pi * f_note * seg), ms=10)

NFFTS = [512, 2048, 8192]
START_N = 1   # slider starts at 2,048

fig = go.Figure()
for i, nfft in enumerate(NFFTS):
    t_g, f_g, S = compute_spectrogram(scale_sig, AUDIO_RATE,
                                      nfft=nfft, hop=nfft // 4, fmax=700)
    fig.add_trace(go.Heatmap(x=t_g, y=f_g, z=S, colorscale='Inferno',
                             showscale=False, visible=(i == START_N)))

steps = []
for i, nfft in enumerate(NFFTS):
    vis = [False] * len(NFFTS)
    vis[i] = True
    frame_ms = nfft / AUDIO_RATE * 1e3
    df_bin = AUDIO_RATE / nfft
    steps.append(dict(method='update',
                      label=f'{nfft} ({frame_ms:.0f} ms, {df_bin:.1f} Hz)',
                      args=[{'visible': vis}]))

fig.update_layout(
    sliders=[dict(active=START_N, currentvalue=dict(prefix='frame length = '), steps=steps)],
    title='Fig. B.6 — One scale at three frame lengths',
    template='simple_white', height=430,
    xaxis_title='t (s)', yaxis_title='frequency (Hz)',
    margin=dict(l=60, r=20, t=60, b=90))
fig.show()


At 512 samples per frame the note changes are crisply timed and each note is a thick, uncertain band. At 8,192 the notes are thin, precise lines whose starts and ends smear into their neighbors. The middle setting, 2,048, is the one used throughout Part A, chosen because tones tens of Hz apart and events tens of ms apart both matter for machine sounds. The softened staircase edges visible in Fig. A.5 have the same origin: each frame that straddles a note change reports both notes at once.

▸ **Key terms: frame, hop, time-frequency tradeoff**


---

## B.6 Analyzing Your Own Recording

Everything in this Part now runs in one direction: from a recording, through sampling and the DFT, to the three views. This section points the toolkit at a recording of your own, with the analysis decisions in your hands. Record a few seconds of sound on your phone, a whistle, a sung vowel, a clap, a running machine, export or convert it to a WAV file, and place it beside this notebook. If no file is found, the cell synthesizes an example recording containing a hummed tone, a clap, and a rising whistle, writes it as `example_recording.wav`, and analyzes that instead, so the section runs either way and switching to your own sound is a one-string edit.

Four parameters at the top of the cell are yours to adjust, and each is one of the chapter's ideas made concrete. The analysis window (`t_start`, `duration`) selects which stretch of the recording to analyze, and its length $T$ sets the frequency resolution through Eq. B.6. The resample rate (`fs_new`) reduces the sample rate by keeping every $n$-th sample, deliberately without the protective filter that real recording hardware applies, so content above the new Nyquist frequency of Eq. B.1 aliases honestly and you can produce the failure of Sect. B.2 on your own sound. The frame length (`nfft`) positions the spectrogram on the time-frequency tradeoff of Sect. B.5.


In [ ]:
# --- Reading and synthesizing recordings ---
import wave as _wave

def load_wav(path):
    """Read a WAV file (PCM 8, 16, 24, or 32 bit, mono or stereo) and return
    (sample_rate, mono_signal) with the signal scaled to the range -1 to 1."""
    with _wave.open(path, 'rb') as wf:
        fs = wf.getframerate()
        n_ch = wf.getnchannels()
        sw = wf.getsampwidth()
        raw = wf.readframes(wf.getnframes())
    if sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(float) - 128.0) / 128.0
    elif sw == 2:
        x = np.frombuffer(raw, dtype='<i2').astype(float) / 32768.0
    elif sw == 3:
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        x = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16)
        x = np.where(x >= 2**23, x - 2**24, x).astype(float) / 2**23
    elif sw == 4:
        x = np.frombuffer(raw, dtype='<i4').astype(float) / 2**31
    else:
        raise ValueError(f'unsupported WAV sample width: {sw} bytes')
    if n_ch > 1:
        x = x.reshape(-1, n_ch).mean(axis=1)
    return fs, x

def make_example_recording(path='example_recording.wav'):
    """Synthesize a 4 s example (hummed tone, clap, rising whistle), write it
    as a 16-bit WAV beside the notebook, and return (sample_rate, signal)."""
    fs = AUDIO_RATE
    t = np.arange(int(4.0 * fs)) / fs
    x = np.zeros_like(t)
    seg = (t >= 0.2) & (t < 1.4)                     # hummed tone, 220 Hz recipe
    for k in range(1, 7):
        x[seg] += (0.5 / k) * np.sin(2 * np.pi * k * 220 * t[seg])
    seg = (t >= 1.7) & (t < 1.9)                     # clap
    rng = np.random.default_rng(26)
    x[seg] += np.exp(-(t[seg] - 1.7) / 0.02) * rng.standard_normal(int(seg.sum())) * 0.8
    seg = (t >= 2.2) & (t < 3.8)                     # rising whistle, 800 to 1600 Hz
    ts = t[seg] - 2.2
    f_w = 800 + 500 * ts
    x[seg] += 0.5 * np.sin(2 * np.pi * np.cumsum(f_w) / fs)
    x = fade(0.9 * x / np.max(np.abs(x)))
    with _wave.open(path, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(fs)
        wf.writeframes((x * 32767).astype('<i2').tobytes())
    return fs, x


In [ ]:
# --- Analyze a recording. Adjust the parameters and rerun. ---
filename = 'my_recording.wav'   # your WAV file; the example is used when absent
t_start = 0.0                   # s, where the analysis window begins
duration = 4.0                  # s, window length; sets the resolution of Eq. B.6
fs_new = None                   # Hz, resample rate, e.g. 4000; None keeps the original
nfft = 2048                     # samples per spectrogram frame, Sect. B.5
fmax = 4000                     # Hz, top of the frequency axes

import os
if os.path.exists(filename):
    fs0, x_rec = load_wav(filename)
    print(f'Loaded {filename}.')
else:
    fs0, x_rec = make_example_recording()
    print(f'{filename} not found; using the synthesized example '
          '(written beside this notebook as example_recording.wav).')

i0 = int(t_start * fs0)
x_rec = x_rec[i0:i0 + int(duration * fs0)]
x_rec = x_rec / (np.max(np.abs(x_rec)) + 1e-12)

fs_rec = float(fs0)
if fs_new is not None and fs_new < fs0:
    stride = int(round(fs0 / fs_new))
    x_rec = x_rec[::stride]                # deliberate, unguarded resampling
    fs_rec = fs0 / stride
elif fs_new is not None:
    print('fs_new is at or above the original rate; keeping the original.')

T_rec = len(x_rec) / fs_rec
f_top = min(fmax, fs_rec / 2)
print(f'Analysis window T = {T_rec:.2f} s at fs = {fs_rec:,.0f} per second: '
      f'Nyquist {fs_rec / 2:,.0f} Hz (Eq. B.1), resolution {1 / T_rec:.2f} Hz (Eq. B.6).')
display(Audio(fade(0.8 * x_rec), rate=int(fs_rec)))

tt, xx = decimate_for_plot(np.arange(len(x_rec)) / fs_rec, x_rec,
                           max_points=6000)          # display only; see Appendix
f_sp, a_sp = compute_spectrum(x_rec, fs_rec, fmax=f_top)
t_g, f_g, S = compute_spectrogram(x_rec, fs_rec, nfft=nfft, hop=nfft // 4, fmax=f_top)

fig = make_subplots(rows=1, cols=3,
                    column_titles=('time domain', 'spectrum', 'spectrogram'))
fig.add_trace(go.Scatter(x=tt, y=xx, mode='lines',
                         line=dict(color='#1f77b4', width=0.7),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=f_sp, y=a_sp, mode='lines',
                         line=dict(color='#1f77b4'), showlegend=False), row=1, col=2)
fig.add_trace(go.Heatmap(x=t_g, y=f_g, z=S, colorscale='Inferno',
                         showscale=False), row=1, col=3)
fig.update_xaxes(title_text='t (s)', row=1, col=1)
fig.update_xaxes(title_text='frequency (Hz)', row=1, col=2)
fig.update_xaxes(title_text='t (s)', row=1, col=3)
fig.update_yaxes(title_text='amplitude', row=1, col=1)
fig.update_yaxes(title_text='amplitude (normalized)', range=[0, 1.05], row=1, col=2)
fig.update_yaxes(title_text='frequency (Hz)', range=[0, f_top], row=1, col=3)
fig.update_layout(title='Fig. B.7 — A recording of your choice in three views',
                  template='simple_white', height=380,
                  margin=dict(l=60, r=20, t=70, b=50))
fig.show()


Read the example, or your own recording, with the chapter's questions. The hummed tone shows a stack of harmonics in the spectrum, the recipe of Sect. B.3; the clap is a brief vertical stripe on the spectrogram with broad content in the spectrum, the temporal cousin of a sharp corner from the Corners and Clicks box; and the whistle draws a rising line that the whole-record spectrum smears into a block, the lesson of Sect. A.8 on your own sound.

Then work the parameters. Shorten `duration` to 0.5 s and watch every spectral peak widen as Eq. B.6 coarsens the resolution. Set `fs_new = 4000` and listen: any content above the new 2,000 Hz Nyquist frequency folds down to an aliased frequency, and the whistle's line bends where it crosses the boundary. Move `nfft` between 512 and 8192 and place the spectrogram where your recording needs it on the time-frequency tradeoff. Each adjustment is a decision a measurement engineer makes before trusting a spectrum, and Eqs. B.1 and B.6 predict the consequence of every one.

<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
Phone voice memo apps commonly export M4A rather than WAV; most phones and computers convert with a share or export option, and the free tool ffmpeg converts anything, including the soundtrack of a video, with one command: <code>ffmpeg -i input.mp4 -ac 1 output.wav</code>. The same chain, microphone to spectrum to spectrogram, runs continuously for live listening in practice; Part C points it at a machining process.
</div>


---

## B.7 The Sound of a Periodic Force

One observation now carries the whole of Part B into Part C. Milling is interrupted cutting: each tooth of the rotating cutter enters the workpiece, cuts, and leaves, and the next tooth repeats the event. The cutting force is therefore a train of similar pulses arriving at a fixed rate, and a pulse train is a periodic signal whose fundamental frequency equals that rate. By Eq. B.2, its content is harmonics at integer multiples of the fundamental frequency, so the sound of a stable milling cut consists of evenly spaced spectral lines. Fig. B.8 shows the principle with a synthetic pulse train whose fundamental frequency is 250 Hz.


In [ ]:
# --- Fig. B.8: a repeating pulse and its spectrum ---
f_rep = 250.0
dur = 1.0
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE
width = 0.0004                                   # each pulse lasts 0.4 ms
phase = (t * f_rep) % 1.0
pulses = np.where(phase < width * f_rep, 1.0, 0.0)

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=('Pulse train (first 20 ms)', 'Spectrum'))
n_show = int(0.02 * AUDIO_RATE)
fig.add_trace(go.Scatter(x=t[:n_show] * 1e3, y=pulses[:n_show], mode='lines',
                         line=dict(color='#1f77b4'), showlegend=False), row=1, col=1)
f_sp, a_sp = compute_spectrum(pulses, AUDIO_RATE, fmax=4000)
fig.add_trace(go.Scatter(x=f_sp, y=a_sp, mode='lines',
                         line=dict(color='#1f77b4'), showlegend=False), row=1, col=2)
fig.update_layout(title='Fig. B.8 — A repeating pulse and its spectrum',
                  template='simple_white', height=380,
                  margin=dict(l=60, r=20, t=70, b=50))
fig.update_xaxes(title_text='t (ms)', row=1, col=1)
fig.update_xaxes(title_text='frequency (Hz)', row=1, col=2)
fig.update_yaxes(title_text='force (normalized)', row=1, col=1)
fig.update_yaxes(title_text='amplitude (normalized)', range=[0, 1.05], row=1, col=2)
fig.show()

print('The pulse train, as sound:')
display(Audio(fade(0.5 * (pulses - pulses.mean()) / np.max(np.abs(pulses))),
              rate=AUDIO_RATE))


The spectrum consists of harmonics: spectral lines at the 250 Hz fundamental and its integer multiples, with heights that roll off gradually because each individual pulse is short. In a milling cut the fundamental frequency is the tooth passing frequency, set by the spindle speed and the number of teeth, and the harmonics it produces are the healthy hum we previewed in Sect. A.2. The frequency-domain treatment of milling in Sect. 4.3.3 of Schmitz and Smith [6] expands the periodic cutting force in exactly this way, as a Fourier series.

Part C applies everything assembled here. We give these harmonics their machining names, simulate the cutting process in the time domain following the straight tooth model of Sect. 4.4 of Schmitz and Smith [6], and listen to what happens when the cut stops being periodic: the self-excited squeal of chatter, arriving at a frequency that is a multiple of no fundamental. The spectrum of this Part becomes the tooth passing family of a cut, the spectrogram becomes the view in which chatter appears between the family lines, and the sonification used here becomes the way the Part C simulation is heard.

▸ **Key terms: pulse train**


---

## Summary

- Sound is a traveling pressure wave; a microphone records pressure against time, and pitch and loudness correspond to the rate and size of the fluctuation.
- Sampling converts the wave into $f_s$ numbers per second; frequencies up to the Nyquist frequency $f_N = f_s/2$ (Eq. B.1) are represented faithfully, and content above it aliases to a lower frequency.
- A periodic signal can be written as a Fourier series (Eq. B.2), a sum of harmonics at multiples of the fundamental frequency; the amplitude recipe is the signal's timbre, and the same synthesis traces closed curves as readily as waveforms.
- The discrete Fourier transform (Eq. B.3) recovers the recipe from $N$ samples with resolution $\Delta f = 1/T$ (Eq. B.6), the FFT computes it quickly, and `compute_spectrum` is this computation with a window and normalization.
- The spectrogram stacks windowed short-time spectra frame by frame; the frame length sets a time-frequency tradeoff, with short frames precise in time and long frames precise in frequency.
- The complete chain applies unchanged to any recording; the analysis window, sample rate, and frame length are the analyst's decisions, and Eqs. B.1 and B.6 predict the consequences of each.
- A repeating force is periodic, so its sound contains harmonics at integer multiples of its fundamental frequency, the form a stable milling cut's sound takes in Part C.


---

## Exercises

**1.** A data acquisition system samples at $f_s = 8{,}000$ per second.
**(a)** What is its Nyquist frequency?
**(b)** Tones at 1,500 Hz, 5,000 Hz, and 7,200 Hz are recorded. Which alias, and at what frequency does each aliased signal appear? The aliased frequency is $|f - f_s \cdot \mathrm{round}(f / f_s)|$.

**2.** Using the explorer of Fig. B.3, find the smallest $K$ at which you can no longer hear a difference from the next step, and compare with a neighbor.
**(a)** At $f_0 = 250$ Hz, what frequency is the highest harmonic you kept?
**(b)** Human hearing ends near 20 kHz. What value of $K$ would place the highest harmonic at that limit, and what does that suggest about the audibility of the terms beyond your answer to (a)?

**3.** In the cell of Fig. B.4, change the recipe amplitudes from $1/k$ to $1/k^2$ for both recipes and rerun.
**(a)** Predict, before running, how the sound will change.
**(b)** Confirm against the spectrum, and describe the change in timbre in one sentence.

**4.** Two tones 20 Hz apart must appear as separate peaks in a spectrum.
**(a)** Using Eq. B.6, what minimum record length is required?
**(b)** For a spectrogram at $f_s = 48{,}000$, what minimum frame length in samples is required, and what does that frame length cost in timing precision?

**5.** In the cell of Fig. B.5, change the text string to your own initials and rerun.
**(a)** Which features of your letters resolve last as $K$ grows?
**(b)** Using the discussion in the Corners and Clicks box, explain the pattern in one sentence.

**6.** Record a whistle that slides from low to high, load it in Sect. B.6, and analyze it.
**(a)** From the spectrogram, what frequency range does your whistle cover?
**(b)** Choose `fs_new` so that the top of your whistle lies above the new Nyquist frequency but the bottom lies below it. Predict, using the aliased-frequency formula from Exercise 1, where the folded portion of the line will appear, then rerun and check.


---

## Appendix: Utility Functions

The code cells of this lesson rely on the helper functions, defined once in the collapsed cell at the top of the notebook (they must be defined before their first use so that Restart and Run All succeeds). They are identical to the functions used in Part A. For reference:

- `fade(x, ms, fs)` applies a short raised-cosine fade-in and fade-out so audio clips start and stop without clicks.
- `compute_spectrum(x, fs, fmax)` returns the amplitude spectrum of a signal on a linear scale, normalized to a peak of one; Sect. B.4 walks through it line by line.
- `compute_spectrogram(x, fs, nfft, hop, fmax)` returns the short-time spectrogram on the same linear, normalized scale; Sect. B.5 describes its frame-by-frame operation.
- `decimate_for_plot(t, x, max_points)` passes plotting data through unchanged by default; set `max_points` to an integer to thin a long record for display, keeping each bin's minimum and maximum so the visual envelope is preserved without aliasing. Computations always use the full-rate data.


---

## References

**[1]** Nyquist, H. (1928). Certain topics in telegraph transmission theory. *Transactions of the AIEE*, 47(2), 617 to 644.

**[2]** Shannon, C. E. (1949). Communication in the presence of noise. *Proceedings of the IRE*, 37(1), 10 to 21.

**[3]** Fourier, J. B. J. (1822). *Theorie analytique de la chaleur*. Paris: Firmin Didot. English translation: *The Analytical Theory of Heat*, Cambridge University Press, 1878.

**[4]** Helmholtz, H. (1863). *Die Lehre von den Tonempfindungen*. English translation by A. J. Ellis from the 4th German edition: *On the Sensations of Tone as a Physiological Basis for the Theory of Music* (1877). Reprinted by Dover Publications, 1954.

**[5]** Cooley, J. W., and Tukey, J. W. (1965). An algorithm for the machine calculation of complex Fourier series. *Mathematics of Computation*, 19(90), 297 to 301. DOI: 10.1090/S0025-5718-1965-0178586-1

**[6]** Schmitz, T. L., and Smith, K. S. (2019). *Machining Dynamics: Frequency Response to Improved Productivity*, 2nd ed. Springer. DOI: 10.1007/978-3-319-93707-6

**[7]** Schmitz, T. L., and Gomez, M. F. *Machining Dynamics: Jupyter Notebook Edition* (in preparation).
